To maintain the integrity of your data, it’s a good idea to reduce duplicate documents in the system. Plagiarism Checker helps us to detect duplicates. The database has N documents. The common structure of each document is title, author names, affiliation, abstract, keywords, Paper content, Conclusion. A Plagiarism Checker reads a new document and adds it to the database if it is not a duplicate of any document in the database. Design a Plagiarism Checker with the following features.

A) Verify if the titles are exactly same (Apply BinaryDistance(u,v), which gives the binary distance between vectors u and v, equal to 0 if they are identical and 1 otherwise.). If same, label the document as duplicate and discard it else proceed to second part of the Checker.

B) Represent documents (Paper content) as term document vectors with weight of a
term in a document computed as [formula given]

C) Identify a document as duplicate if the similarity of the document is more than the threshold α. (α=0.85). Apply cosine similarity for similarity computations.

D) Identify k-Shingles (N grams) in the documents and apply Jaccard similarity to identify duplicates

E) Apply probability retrieval model (Okapi BM25) to identify duplicates

Sample dataset

D1: Information requirement: query considers the user feedback as information
requirement to search.

D2: Information retrieval: query depends on the model of information retrieval

used.
D3: Prediction problem: Many problems in information retrieval can be viewed as

prediction problems

D4: Search: A search engine is one of applications of information retrieval models.

New documents

D5: Feedback: feedback is typically used by the system to modify the query and

improve prediction

D6: information retrieval: ranking in information retrieval algorithms depends on

user query

In [4]:
class Document:
    def __init__(self, doc_id, title, content):
        self.doc_id =  doc_id
        self.title = title
        self.content = content

database = [
    Document("D1", "Information requirement", "query considers the user feedback as information requirement to search."),
    Document("D2", "Information retrieval", "query depends on the model of information retrieval used."),
    Document("D3", "Prediction problem", "Many problems in information retrieval can be viewed as prediction problems"),
    Document("D4", "Search", "A search engine is one of applications of information retrieval models.")
]

new_docs = [
    Document("D5", "Feedback", "feedback is typically used by the system to modify the query and improve prediction"),
    Document("D6", "information retrieval", "ranking in information retrieval algorithms depends on user query")
]

In [6]:
import re
import math
from collections import Counter

def binary_distance(u,v):
  if u.strip().lower() == v.strip().lower():
    return 0
  return 1


def plagiarism_checker_binary_dist(new_doc, database):
  print(f"\nChecking Document: '{new_doc.title}'")
  for doc in database:
    dist = binary_distance(doc.title, new_doc.title)
    if dist==0:
      print(f"Match Found: Same title as '{doc.title}' (Distance: 0)")
      return True

  print("No match found.. Proceed next step..")
  return False

# -------------------------------------------------------------

def preprocess(text):
    text = text.lower()
    words = re.findall(r'\w+', text)
    return words


def get_database_idf(database):
    processed_docs = [preprocess(doc.content) for doc in database]
    N = len(database)
    vocab = set(word for doc in processed_docs for word in doc)

    idf_table = {}
    for term in vocab:
        df_k = sum(1 for doc in processed_docs if term in doc)
        # log10((N + 1) / (0.5 + df_k))
        idf_table[term] = math.log10((N + 1) / (0.5 + df_k))
    return idf_table


def calculate_weights(doc_content, idf_table):
    words = preprocess(doc_content)
    counts = Counter(words)
    length = len(words)
    vector = {}
    for term, idf_weight in idf_table.items():
        tf = counts.get(term, 0) / length
        vector[term] = tf * idf_weight
    return vector


def get_shingles(text, k=3): # N-grams
    words = preprocess(text)
    return set([" ".join(words[i:i+k]) for i in range(len(words)-k+1)])

In [7]:
def cosine_similarity(vec1, vec2):
  print(f"Vector 1: {vec1}")
  print(f"Vector 2: {vec2}")
  intersection = set(vec1.keys()) & set(vec2.keys())
  numerator = sum([vec1[x] * vec2[x] for x in intersection])
  sum1 = sum([vec1[x] ** 2 for x in list(vec1.keys())])
  sum2 = sum([vec2[x] ** 2 for x in list(vec2.keys())])
  denominator = math.sqrt(sum1) * math.sqrt(sum2)
  if not denominator:
    return 0.0
  else:
    return float(numerator) / denominator

In [8]:
def get_jaccard_similarity(set1, set2):
    if not set1 or not set2: return 0.0
    return len(set1 & set2) / len(set1 | set2)

In [9]:
new_valid_docs = []

for doc in new_docs:
  isPlagiarised = plagiarism_checker_binary_dist(doc, database)
  if not isPlagiarised:
    new_valid_docs.append(doc)

print("-"*30)

idf_table = get_database_idf(database)
print(f"\nIDF Table: {idf_table}")

database_vectors = {}
for doc in database:
  database_vectors[doc.doc_id] = calculate_weights(doc.content, idf_table)
new_docs_vectors = {}
for doc in new_valid_docs:
  new_docs_vectors[doc.doc_id] = calculate_weights(doc.content, idf_table)

print(f"\nDatabase Vectors: {database_vectors}")
print(f"\nNew Docs Vectors: {new_docs_vectors}")

print("-"*30)

alpha = 0.85
print("\nCosine Similarity Scores:")
for new_doc_id, vector_A in new_docs_vectors.items():
  for doc_id, vector_B in database_vectors.items():
    similarity = cosine_similarity(vector_A, vector_B)
    print(f" - Cosine similarty of {doc_id} and {new_doc_id}: ", similarity)
    if similarity > alpha:
      print(f"Duplicate detected! - {doc_id} and {new_doc_id} are similar.")

print("-"*30)

print("\nJaccard Similarity Scores:")
database_shingles = {}
new_doc_shingles = {}
k=3

for doc in database:
  database_shingles[doc.doc_id] = get_shingles(doc.content,k)
for new_doc in new_valid_docs:
  new_doc_shingles[new_doc.doc_id] = get_shingles(new_doc.content,k)

for new_doc_id, shingles_A in new_doc_shingles.items():
  for doc_id, shingles_B in database_shingles.items():
    similarity = get_jaccard_similarity(shingles_A, shingles_B)
    print(f" - Jaccard similarty of {doc_id} and {new_doc_id}: ", similarity)
    if similarity > alpha:
      print(f"Duplicate detected! - {doc_id} and {new_doc_id} are similar.")

print("-"*30)




Checking Document: 'Feedback'
No match found.. Proceed next step..

Checking Document: 'information retrieval'
Match Found: Same title as 'Information retrieval' (Distance: 0)
------------------------------

IDF Table: {'of': 0.3010299956639812, 'query': 0.3010299956639812, 'used': 0.5228787452803376, 'as': 0.3010299956639812, 'models': 0.5228787452803376, 'applications': 0.5228787452803376, 'the': 0.3010299956639812, 'information': 0.04575749056067514, 'depends': 0.5228787452803376, 'to': 0.5228787452803376, 'prediction': 0.5228787452803376, 'one': 0.5228787452803376, 'user': 0.5228787452803376, 'many': 0.5228787452803376, 'on': 0.5228787452803376, 'problems': 0.5228787452803376, 'considers': 0.5228787452803376, 'requirement': 0.5228787452803376, 'model': 0.5228787452803376, 'engine': 0.5228787452803376, 'search': 0.3010299956639812, 'can': 0.5228787452803376, 'a': 0.5228787452803376, 'retrieval': 0.15490195998574316, 'viewed': 0.5228787452803376, 'be': 0.5228787452803376, 'is': 0.52

In [ ]:
pip install rank_bm25

In [ ]:
from rank_bm25 import BM25Okapi

tokenized_corpus = [preprocess(doc.content) for doc in database]
bm25 = BM25Okapi(tokenized_corpus)

print("BM25 Scores")
for new_doc in new_valid_docs:
    query_tokens = preprocess(new_doc.content)
    doc_scores = bm25.get_scores(query_tokens)
    for i, score in enumerate(doc_scores):
        db_id = database[i].doc_id
        print(f"BM25 score between {new_doc.doc_id} and {db_id}: {score:.4f}")

BM25 Scores
BM25 score between D5 and D1: 1.7134
BM25 score between D5 and D2: 0.8965
BM25 score between D5 and D3: 0.8203
BM25 score between D5 and D4: 0.8203
